# 02 · Model comparison under k-fold cross validation

Every model is evaluated on **the same fold assignment**, which is what makes
the paired significance test at the end legitimate.

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src.config import Config

# `memory` keeps this session's intermediate frames out of the file system;
# switch to "file" to persist them under data/processed/.
CONFIG = Config(persistence_backend="memory")
CONFIG.ensure_directories()
CONFIG.label_columns

In [ ]:
from src.dataio import load_training_data, raw_files_available

# allow_synthetic keeps the notebook runnable before the Kaggle download;
# every number below is only meaningful once raw_files_available() is True.
train = load_training_data(config=CONFIG, allow_synthetic=True)
print("real dataset:", raw_files_available(), "| rows:", len(train))
train.head(3)

In [ ]:
from src.dataio import label_matrix, stratified_subsample, texts

# Drop to a stratified subsample while iterating; remove for the final run.
SAMPLE_SIZE = 20_000
frame = stratified_subsample(train, SAMPLE_SIZE, config=CONFIG) if len(train) > SAMPLE_SIZE else train

comment_text = texts(frame, CONFIG).tolist()
y = label_matrix(frame, CONFIG)
label_names = list(CONFIG.label_columns)
print(f"{len(frame)} comments, positive rate per label:")
print(pd.Series(y.mean(axis=0), index=label_names).round(4))

## Fold construction

`iterative` means the `iterative-stratification` package balanced all six labels
across the folds; `label_powerset` is the fallback. **Record which one ran** —
they are not interchangeable.

In [ ]:
from src.evaluation import make_multilabel_folds

folds, strategy = make_multilabel_folds(y, n_splits=CONFIG.n_splits,
                                        random_state=CONFIG.random_state)
print("strategy:", strategy, "| folds:", len(folds))

# Check the rare labels survived the split.
fold_rates = pd.DataFrame(
    [y[validation].mean(axis=0) for _, validation in folds],
    columns=label_names, index=[f"fold {i}" for i in range(1, len(folds) + 1)],
)
display(fold_rates.style.format("{:.4f}"))

## Cross-validating every model

In [ ]:
from src.evaluation import cross_validate_model
from src.models import build_model, lightweight_models

MODELS = lightweight_models()   # add "distilbert" once requirements-dl.txt is installed
results = {}

for name in MODELS:
    print(f"--- {name} ---")
    results[name] = cross_validate_model(
        model_factory=lambda name=name: build_model(name, label_names=label_names),
        texts=comment_text, y=y, label_names=label_names,
        folds=folds, strategy=strategy, random_state=CONFIG.random_state,
    )
    print(results[name].summary[["mean_column_auc_mean", "macro_f1_mean"]].round(4).to_dict())

## Leaderboard

In [ ]:
leaderboard = (
    pd.DataFrame([result.summary for result in results.values()])
    .sort_values("mean_column_auc_mean", ascending=False)
)
display(
    leaderboard[[
        "mean_column_auc_mean", "mean_column_auc_std", "mean_column_pr_auc_mean",
        "macro_f1_mean", "micro_f1_mean", "subset_accuracy_mean", "fit_seconds_mean",
    ]].round(4)
)

fig, ax = plt.subplots(figsize=(7.5, 3.5))
ax.barh(leaderboard.index, leaderboard["mean_column_auc_mean"],
        xerr=leaderboard["mean_column_auc_std"], color="#4C72B0", capsize=4)
ax.axvline(0.5, color="grey", ls="--", lw=1)
ax.text(0.505, -0.4, "chance", color="grey", fontsize=8)
ax.set(xlabel="mean column-wise ROC-AUC", xlim=(0.45, 1.0), title="Model comparison (5-fold)")
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "leaderboard.png", dpi=200)

## Per-label breakdown

The aggregate hides the interesting part: the rare labels are where the models
actually differ.

In [ ]:
best = leaderboard.index[0]
display(results[best].label_summary.round(4))

comparison = pd.DataFrame(
    {name: result.label_summary["pr_auc"] for name, result in results.items()}
)
fig, ax = plt.subplots(figsize=(8, 3.5))
comparison.plot.bar(ax=ax, width=0.8)
ax.set(ylabel="PR-AUC", xlabel="", title="Average precision per label")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.xticks(rotation=30, ha="right")
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "pr_auc_per_label.png", dpi=200)

## Is the difference real?

With five folds the Wilcoxon test bottoms out at p = 0.0625, so no pairwise
difference can reach the 5% level however large it is. Report the effect size
next to the p-value, and increase the number of folds if the claim matters.

In [ ]:
from src.evaluation import comparison_matrix, paired_comparison

matrix = comparison_matrix({name: result.fold_scores() for name, result in results.items()})
print(f"Bonferroni-corrected alpha = {matrix.attrs['bonferroni_alpha']:.4f}")
display(matrix.round(4))

runner_up = leaderboard.index[1]
best_vs_second = paired_comparison(
    results[best].fold_scores(), results[runner_up].fold_scores(), best, runner_up
)
pd.Series(best_vs_second.to_dict())

## Cost against accuracy

The practical question a deployment team asks.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(leaderboard["fit_seconds_mean"], leaderboard["mean_column_auc_mean"], s=70)
for name, row in leaderboard.iterrows():
    ax.annotate(name, (row["fit_seconds_mean"], row["mean_column_auc_mean"]),
                textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set(xscale="log", xlabel="mean fit time per fold (s, log scale)",
       ylabel="mean column-wise AUC", title="Accuracy against training cost")
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "cost_vs_accuracy.png", dpi=200)

## Persist the results

Stored through the artifact repository, so this cell is backend-agnostic.

In [ ]:
from src.repository import get_artifact_repository

artifacts = get_artifact_repository(Config(persistence_backend="file"))
for name, result in results.items():
    artifacts.save(f"notebook_{name}", result.to_dict())
artifacts.save("notebook_leaderboard", leaderboard.reset_index(names="model").to_dict("records"))
artifacts.list_artifacts()